In [9]:
from pyspark.sql.functions import col

env_files = {
    "ghg_per_kg": "ghg-per-kg-poore.csv",
    "land_per_kg": "land-use-per-kg-poore.csv",
    "water_per_kg": "water-withdrawals-per-kg-poore.csv",
    "ghg_per_100g_protein": "ghg-per-protein-poore.csv",
    "land_per_100g_protein": "land-use-protein-poore.csv",
    "water_per_100g_protein": "water-per-protein-poore.csv",
}

env = None
for metric, filename in env_files.items():
    d = spark.read.csv(f"Files/food-environmental-impact/{filename}", header=True)
    value_col = d.columns[-1]   # the metric is the last column in each file
    d = d.select(col("Entity").alias("owid_entity"),
                 col(f"`{value_col}`").cast("double").alias(metric))
    env = d if env is None else env.join(d, "owid_entity", "outer")

env.write.format("delta").mode("overwrite").saveAsTable("silver_env_impact")
print(env.count(), "environmental rows saved")
env.orderBy("owid_entity").show(40, truncate=False)

StatementMeta(, 8de2dff5-28e6-43f2-99fc-29062ccff8e0, 11, Finished, Available, Finished, False)

39 environmental rows saved
+-----------------+----------+-----------+------------+--------------------+---------------------+----------------------+
|owid_entity      |ghg_per_kg|land_per_kg|water_per_kg|ghg_per_100g_protein|land_per_100g_protein|water_per_100g_protein|
+-----------------+----------+-----------+------------+--------------------+---------------------+----------------------+
|Apples           |0.43      |0.63       |180.1       |14.333333           |21.0                 |6003.3335             |
|Bananas          |0.86      |1.93       |114.5       |9.555555            |21.444445            |1272.2222             |
|Barley           |1.18      |1.11       |17.1        |NULL                |NULL                 |NULL                  |
|Beef (beef herd) |99.48     |326.21     |1451.2      |49.889668           |163.5958             |727.7833              |
|Beef (dairy herd)|33.3      |43.24      |2714.3      |16.8693             |21.904762            |1375.0253           

In [10]:
from pyspark.sql.functions import col, count, avg, expr, when, sum as spark_sum, round as spark_round

nutri_pts = expr("case nutriscore_grade when 'a' then 1 when 'b' then 2 when 'c' then 3 when 'd' then 4 when 'e' then 5 end")
is_direct = col("match_quality") == "direct"

prod = spark.table("silver_food_products").filter(~col("nutrients_inconsistent"))

check = prod.groupBy("food_category").agg(
    count("*").alias("n_all"),
    spark_sum(when(is_direct, 1).otherwise(0)).alias("n_direct"),
    spark_round(avg("saturated_fat"), 1).alias("satfat_all"),
    spark_round(avg(when(is_direct, col("saturated_fat"))), 1).alias("satfat_direct"),
    spark_round(avg(nutri_pts), 2).alias("nutri_all"),
    spark_round(avg(when(is_direct, nutri_pts)), 2).alias("nutri_direct"),
).withColumn("pct_direct", spark_round(col("n_direct") / col("n_all") * 100, 0))

check.orderBy("pct_direct").show(40, truncate=False)

StatementMeta(, 8de2dff5-28e6-43f2-99fc-29062ccff8e0, 12, Finished, Available, Finished, False)

+----------------+-----+--------+----------+-------------+---------+------------+----------+
|food_category   |n_all|n_direct|satfat_all|satfat_direct|nutri_all|nutri_direct|pct_direct|
+----------------+-----+--------+----------+-------------+---------+------------+----------+
|Tomatoes        |13063|0       |0.4       |NULL         |2.06     |NULL        |0.0       |
|Prawns (farmed) |2989 |0       |0.7       |NULL         |2.21     |NULL        |0.0       |
|Fish (farmed)   |38793|0       |2.3       |NULL         |2.75     |NULL        |0.0       |
|Wheat & Rye     |3929 |566     |0.4       |0.7          |1.31     |1.76        |14.0      |
|Potatoes        |5716 |2142    |2.6       |0.5          |3.17     |1.97        |37.0      |
|Beef            |8256 |6961    |4.1       |4.0          |3.23     |3.07        |84.0      |
|Oatmeal         |3171 |2654    |1.4       |1.3          |1.4      |1.31        |84.0      |
|Pig Meat        |14099|11920   |6.3       |6.5          |4.08     |4.

In [11]:
from pyspark.sql.functions import (col, avg, count, expr, explode, split, when,
                                   min as spark_min, max as spark_max, sum as spark_sum, round as spark_round)

MIN_DIRECT = 500   # use direct matches only if a category has at least this many

nutrients = ["energy_kcal", "proteins", "fat", "saturated_fat", "carbohydrates", "sugars", "fiber", "salt"]
metrics = ["ghg_per_kg", "land_per_kg", "water_per_kg",
           "ghg_per_100g_protein", "land_per_100g_protein", "water_per_100g_protein"]

prod_all = spark.table("silver_food_products").filter(~col("nutrients_inconsistent"))
cats = spark.table("silver_food_category")
env = spark.table("silver_env_impact")

basis = (
    prod_all.groupBy("food_category")
    .agg(count("*").alias("n_products_all"),
         spark_sum(when(col("match_quality") == "direct", 1).otherwise(0)).alias("n_products_direct"))
    .withColumn("basis_used", when(col("n_products_direct") >= MIN_DIRECT, "direct_only").otherwise("all"))
    .withColumn("pct_direct", spark_round(col("n_products_direct") / col("n_products_all") * 100, 0))
)

prod = (
    prod_all.join(basis.select("food_category", "basis_used"), "food_category")
    .filter((col("basis_used") == "all") | (col("match_quality") == "direct"))
)

nutrition = prod.groupBy("food_category").agg(
    count("*").alias("n_products"),
    *[avg(c).alias(f"avg_{c}") for c in nutrients],
    *[count(c).alias(f"n_{c}") for c in nutrients],
    avg(expr("case nutriscore_grade when 'a' then 1 when 'b' then 2 when 'c' then 3 "
             "when 'd' then 4 when 'e' then 5 end")).alias("avg_nutriscore_points"),
    avg(when(col("nutriscore_grade").isin("a", "b"), 1.0)
        .when(col("nutriscore_grade").isNotNull(), 0.0)).alias("share_nutriscore_a_or_b"),
    count("nutriscore_grade").alias("n_nutriscore"),
    avg("nova_group").alias("avg_nova"),
    avg(when(col("nova_group") == 4, 1.0)
        .when(col("nova_group").isNotNull(), 0.0)).alias("share_ultra_processed"),
    count("nova_group").alias("n_nova"),
)

cat_entities = cats.select("food_category", explode(split("owid_entities", ";")).alias("owid_entity"))
environment = (
    cat_entities.join(env, "owid_entity", "left")
    .groupBy("food_category")
    .agg(*[f(m).alias(f"{m}_{suffix}")
           for m in metrics
           for f, suffix in [(spark_min, "low"), (avg, "mid"), (spark_max, "high")]])
)

profile = (cats.join(nutrition, "food_category", "left")
               .join(environment, "food_category", "left")
               .join(basis, "food_category", "left")
               .fillna({"n_products": 0, "n_products_all": 0, "n_products_direct": 0, "basis_used": "none"}))

(profile.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable("gold_category_profile"))

(spark.table("gold_category_profile")
   .select("food_category", "basis_used", "n_products", "n_products_all", "pct_direct",
           spark_round("avg_saturated_fat", 1).alias("sat_fat"),
           spark_round("avg_nutriscore_points", 2).alias("nutriscore_pts"))
   .orderBy("pct_direct")
   .show(40, truncate=False))

StatementMeta(, 8de2dff5-28e6-43f2-99fc-29062ccff8e0, 13, Finished, Available, Finished, False)

+----------------+-----------+----------+--------------+----------+-------+--------------+
|food_category   |basis_used |n_products|n_products_all|pct_direct|sat_fat|nutriscore_pts|
+----------------+-----------+----------+--------------+----------+-------+--------------+
|Cassava         |none       |0         |0             |NULL      |NULL   |NULL          |
|Root Vegetables |none       |0         |0             |NULL      |NULL   |NULL          |
|Maize           |none       |0         |0             |NULL      |NULL   |NULL          |
|Barley          |none       |0         |0             |NULL      |NULL   |NULL          |
|Cane Sugar      |none       |0         |0             |NULL      |NULL   |NULL          |
|Beet Sugar      |none       |0         |0             |NULL      |NULL   |NULL          |
|Fish (farmed)   |all        |38793     |38793         |0.0       |2.3    |2.75          |
|Tomatoes        |all        |13063     |13063         |0.0       |0.4    |2.06          |

In [12]:
from pyspark.sql.functions import col, round as spark_round, when

profile = spark.table("gold_category_profile")
alt = spark.table("silver_plant_alternatives")

env_cols = ["ghg_per_kg", "land_per_kg", "water_per_kg",
            "ghg_per_100g_protein", "land_per_100g_protein", "water_per_100g_protein"]
nut_cols = ["energy_kcal", "proteins", "fat", "saturated_fat", "carbohydrates", "sugars", "fiber", "salt"]

a = profile.select(
    col("food_category").alias("animal_category"),
    col("n_products").alias("animal_n_products"),
    *[col(f"{m}_mid").alias(f"animal_{m}") for m in env_cols],
    *[col(f"avg_{n}").alias(f"animal_{n}") for n in nut_cols],
    col("avg_nutriscore_points").alias("animal_nutriscore_pts"),
)
p = profile.select(
    col("food_category").alias("plant_category"),
    col("n_products").alias("plant_n_products"),
    *[col(f"{m}_mid").alias(f"plant_{m}") for m in env_cols],
    *[col(f"avg_{n}").alias(f"plant_{n}") for n in nut_cols],
    col("avg_nutriscore_points").alias("plant_nutriscore_pts"),
)

swaps = alt.join(a, "animal_category").join(p, "plant_category")

# per 100g protein when both sides have it, otherwise per kg
swaps = swaps.withColumn(
    "comparison_basis",
    when(col("animal_ghg_per_100g_protein").isNotNull() & col("plant_ghg_per_100g_protein").isNotNull(),
         "per_100g_protein").otherwise("per_kg"))

for m in ["ghg", "land", "water"]:
    animal_v = when(col("comparison_basis") == "per_100g_protein",
                    col(f"animal_{m}_per_100g_protein")).otherwise(col(f"animal_{m}_per_kg"))
    plant_v = when(col("comparison_basis") == "per_100g_protein",
                   col(f"plant_{m}_per_100g_protein")).otherwise(col(f"plant_{m}_per_kg"))
    swaps = (swaps.withColumn(f"animal_{m}", animal_v)
                  .withColumn(f"plant_{m}", plant_v)
                  .withColumn(f"{m}_reduction_pct", spark_round((animal_v - plant_v) / animal_v * 100, 0)))

for n in nut_cols + ["nutriscore_pts"]:
    swaps = swaps.withColumn(f"diff_{n}", spark_round(col(f"plant_{n}") - col(f"animal_{n}"), 2))

swaps.write.format("delta").mode("overwrite").saveAsTable("gold_plant_swaps")

(spark.table("gold_plant_swaps")
   .select("animal_category", "plant_category", "rank", "comparison_basis",
           "ghg_reduction_pct", "land_reduction_pct", "water_reduction_pct",
           "diff_saturated_fat", "diff_fiber", "diff_proteins", "diff_nutriscore_pts")
   .orderBy("animal_category", "rank")
   .show(25, truncate=False))

StatementMeta(, 8de2dff5-28e6-43f2-99fc-29062ccff8e0, 14, Finished, Available, Finished, False)

+---------------+--------------+----+----------------+-----------------+------------------+-------------------+------------------+----------+-------------+-------------------+
|animal_category|plant_category|rank|comparison_basis|ghg_reduction_pct|land_reduction_pct|water_reduction_pct|diff_saturated_fat|diff_fiber|diff_proteins|diff_nutriscore_pts|
+---------------+--------------+----+----------------+-----------------+------------------+-------------------+------------------+----------+-------------+-------------------+
|Beef           |Tofu          |1   |per_100g_protein|94.0             |98.0              |91.0               |-2.67             |1.66      |-7.69        |-1.45              |
|Beef           |Peas          |2   |per_100g_protein|99.0             |96.0              |83.0               |-3.7              |5.46      |-14.95       |-1.78              |
|Beef           |Other Pulses  |3   |per_100g_protein|97.0             |92.0              |81.0               |-3.63    

In [13]:
from pyspark.sql.functions import col, row_number, when, lower, trim, xxhash64
from pyspark.sql.window import Window

prod = (spark.table("silver_food_products")
        .filter(~col("nutrients_inconsistent") & col("product_name").isNotNull())
        .filter(col("product_name").rlike("^[A-Za-z]")))

completeness = sum(when(col(c).isNotNull(), 1).otherwise(0) for c in
    ["nutriscore_grade", "nova_group", "energy_kcal", "proteins", "fat",
     "saturated_fat", "carbohydrates", "sugars", "fiber", "salt"])

prod = (prod.withColumn("completeness", completeness)
            .withColumn("name_key", lower(trim(col("product_name")))))

# one row per product name within a category
w_name = Window.partitionBy("food_category", "name_key").orderBy(col("completeness").desc(), "code")
prod = prod.withColumn("rn_name", row_number().over(w_name)).filter(col("rn_name") == 1)

# up to 10 per category: direct matches first, most complete first, then a stable pseudo-random order
w = Window.partitionBy("food_category").orderBy(
    when(col("match_quality") == "direct", 0).otherwise(1),
    col("completeness").desc(),
    xxhash64("code"))

examples = (prod.withColumn("rn", row_number().over(w))
                .filter(col("rn") <= 10)
                .select("food_category", "code", "product_name", "match_quality", "nutriscore_grade",
                        "nova_group", "vegan_status", "energy_kcal", "proteins", "fat",
                        "saturated_fat", "carbohydrates", "sugars", "fiber", "salt"))

examples.write.format("delta").mode("overwrite").saveAsTable("gold_product_examples")
print(spark.table("gold_product_examples").count(), "example products saved")
spark.table("gold_product_examples").filter("food_category = 'Beef'") \
     .select("product_name", "nutriscore_grade", "saturated_fat").show(10, truncate=False)

StatementMeta(, 8de2dff5-28e6-43f2-99fc-29062ccff8e0, 15, Finished, Available, Finished, False)

310 example products saved
+----------------------------------------------------------------+----------------+-------------+
|product_name                                                    |nutriscore_grade|saturated_fat|
+----------------------------------------------------------------+----------------+-------------+
|Haché pur boeuf surgelé 400g                                    |c               |6.2          |
|Steak Strips                                                    |e               |0.0          |
|British beef kebabs with sweet chilli seasoning                 |d               |7.4          |
|Plat-de-côtes avec os sous vide Halal                           |c               |5.72         |
|Langue de boeuf - Sauce madère au sel de Guérande               |d               |5.5          |
|Tendre & Moelleux 15% M.G.                                      |d               |8.0          |
|Viande bovine - Cuisson sur pierre/plancha, Nouvelle agriculture|a               |1.5     

In [1]:
import os

os.makedirs("/lakehouse/default/Files/gold_export", exist_ok=True)

tables = ["gold_category_profile", "gold_plant_swaps", "gold_product_examples",
          "silver_food_category", "silver_category_mapping",
          "silver_plant_alternatives", "silver_env_impact"]

for t in tables:
    spark.table(t).toPandas().to_csv(f"/lakehouse/default/Files/gold_export/{t}.csv", index=False)
    print("saved", t)

StatementMeta(, 080fb336-7e8a-41d5-9125-efe6900c0ca3, 3, Finished, Available, Finished, False)

saved gold_category_profile
saved gold_plant_swaps
saved gold_product_examples
saved silver_food_category
saved silver_category_mapping
saved silver_plant_alternatives
saved silver_env_impact
